In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# %pip install tiktoken
import tiktoken
import numpy as np

In [2]:
#Building the Dataset
f = open('tiny-shakespeare.txt')
text = f.read()

print("Length of the Dataset:", len(text), "\n")
print(text[:100]) #first 100 characters

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'\nVocabulary Size: {vocab_size}')
print('-'.join(chars))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i: ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ary: ''.join([itos[i] for i in ary])
raw_text = "ben Onur"
print(f'\nRaw Text:  {raw_text}')
token_list = encode(raw_text)
print("Token List:", token_list)
decoded = decode(token_list)
print("decoded:", decoded)

# Compare with OpenAI byte-pair encoding (BPE)
# Tiktoken shows that there is a trade-off between the length of the encoding and the amount of tokens.
# We can have short sequences of tokens with very large vocabulary, or we can just as well have long sequences of tokens with a small vocabulary.
# The BPE approach is widely used for NLP tasks
enc = tiktoken.get_encoding('gpt2')

token_list_BPE = enc.encode(raw_text)
print("token_list_BPE:", token_list_BPE) # BPE returns fewer tokens than the character encoding
print(enc.decode(enc.encode(raw_text)))

print(enc.n_vocab) # total amount of tokens in the vocabulary


Length of the Dataset: 1115394 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

Vocabulary Size: 65

- -!-$-&-'-,---.-3-:-;-?-A-B-C-D-E-F-G-H-I-J-K-L-M-N-O-P-Q-R-S-T-U-V-W-X-Y-Z-a-b-c-d-e-f-g-h-i-j-k-l-m-n-o-p-q-r-s-t-u-v-w-x-y-z

Raw Text:  ben Onur
Token List: [40, 43, 52, 1, 27, 52, 59, 56]
decoded: ben Onur
token_list_BPE: [11722, 1550, 333]
ben Onur
50257


In [3]:
# Encode the text into a tensor of integers
encoded_data = encode(text)
data = torch.tensor(encoded_data, dtype = torch.long)
print(f'Total size: {data.shape} elements of type {data.dtype}')
print('First 10 tokens from the dataset:', data[:10])

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

torch.manual_seed(1337)
#mini-batching.
def get_batch(encoded_data, block_size, batch_size ):
    ix = torch.randint(0, len(encoded_data) - block_size, (batch_size, ))

    X = torch.stack([encoded_data[i:i+block_size] for i in ix], dim=0)
    Y = torch.stack([encoded_data[i+1:i + block_size + 1] for i in ix], dim=0)

    return X,Y      

batch_size = 4
block_size = 8
xb, yb = get_batch(train_data, block_size, batch_size)
# Print the shape of the batch and the actual data
print('inputs shape: ', xb.shape)
print(xb,'\n')
print('targets shape: ', yb.shape)
print(yb, '\n')

for b in range(batch_size):# batch dimension, number of sequences in the batch (batch_size)
    for ci in range(block_size):# time dimension, number of tokens in the sequence  (block_size)
        input = xb[b, :ci+1]# context means prompt, taking the first t+1 tokens from the b-th sequence in the batch
        output = yb[b, ci] # we take the t-th token from the b-th sequence in the batch for the target (the token we want to predict)
        print(f'for Input {input}, predict the output {output}') # This context <-> target pair is what we feed to the model, it's variable in length
        


Total size: torch.Size([1115394]) elements of type torch.int64
First 10 tokens from the dataset: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
inputs shape:  torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]]) 

targets shape:  torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]]) 

for Input tensor([24]), predict the output 43
for Input tensor([24, 43]), predict the output 58
for Input tensor([24, 43, 58]), predict the output 5
for Input tensor([24, 43, 58,  5]), predict the output 57
for Input tensor([24, 43, 58,  5, 57]), predict the output 1
for Input tensor([24, 43, 58,  5, 57,  1]), predict the output 46
for Input tensor([24, 43, 58,  5, 57,  1, 46]), predict the output 43
for Input tensor([24, 43, 58,  5, 57,  

In [ ]:
# Embedding Layer
# sub-batching using mini-batches
# yb[i, j] is the next token after xb[i, j]

torch.manual_seed(1337)

#inheritance from nn.Module
class BiagramLM(nn.Module):

    def __init__(self, vocab_size, block_size, batch_size = 32):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.batch_size = batch_size
        # Embedding the vocabulary
        # Every one of the vocab_size tokens is represented by a vector of size vocab_size
        # With embedding_dim = vocab_size, the "embedding" isn't an embedding at all — it's a 65 × 65 bigram score matrix
        self.embed = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.vocab_size) # 65 unique 65-dim vectors
        print("Embedding Shape:", self.vocab_size, "X", self.vocab_size)

    def forward(self, idx, targets=None): 
        # idx is of shape (batch_size, block_size)
        # targets is of shape (batch_size, block_size)
        # Embed the input indices, shape is now (batch_size, block_size, embedding_dim=vocab_size) (B, T, C)     
        logits = self.embed(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape # B = batch_size, T = block_size, C = embedding_dim = vocab_size
            logits = logits.view(B*T, C) # Flatten B and T dimensions
            targets = targets.view(B*T) # # Flatten B and T dimensions (targets contains the next token's index for each input sequence in the batch)
            loss = F.cross_entropy(logits, targets) # Calculating cross entropy loss across all tokens in the batch (using targets to plug out the correct token for each input sequence)
        return logits, loss    

    # Generate new tokens based on respective last token of a sequence
    def generate(self, idx, max_new_tokens):
        # Repeated over n times, it will:
        #   forward pass through the model with tokens xb to get logits
        #   disregard everything but the last token of xb
        #   calculate the probability of each possible token in the vocabulary to be the token after this last xb token; this is done with F.softmax
        #   sample a token from the probability distribution with torch.multinomial, this returns an index of the token that we can use to look up the token itself in the vocabulary if we wanted
        #   append the sampled token to the tokens xb
        #   repeat
        for i in range(max_new_tokens):
            # Forward pass (this is the forward function) with the current sequence of characters idx, results in (B, T, C)
            #idx_cond = idx[:, -self.block_size:]        # never exceed the context window

            logits, _ = self.forward(idx) #self(idx_cond) (logits = embed(idx))
            # Focus on the last token from the logits (B, T, C) -> (B, C) (C = last token's embedding for every batch)
            logits_last_token = logits[:, -1, :]

            # Calculate the probability distribution for the next token based on this last token, results in (B, C)
            probs = F.softmax(logits_last_token, dim=1)
            # Sample the next token (B, 1), the token with the highest probability is sampled most likely
            idx_next = torch.multinomial(probs, num_samples=1) #Yes — idx_next is a token id, which is the row index in the embedding look-up table
            # Add the new token to the sequence (B, T+1) for the next iteration
            idx = torch.cat((idx, idx_next), dim=1)

            if i % 7 == 0:
                print("Input idx:", i+1, idx)
                print("Logits Last Token:", logits_last_token, "Shape:", logits_last_token.shape, "idx_next:", idx_next, "logits Shape:", logits.shape)
        return idx

    def train(self):
        # Create a PyTorch Optimizer
        # Instantiate AdamW optimizer with the model parameters (weights) 
        # and a learning rate of 0.001 (often used value for *small* networks)
        opt = torch.optim.AdamW(self.parameters(), lr=1e-3)       

        losses = []

        # Train for 10000 steps/batches
        for steps in range(10000):
            xb, yb = get_batch(train_data, self.block_size, self.batch_size) # Sample a batch of data
            logits, loss = self.forward(xb, yb)                # Forward pass, calculate the loss
            if loss is not None:
                loss.backward()                         # Backprop with PyTorch's autograd 
                                                    # (effectively just updating the logits/the embedding vectors)
            opt.step()                              # Update the weights
            opt.zero_grad()                         # Set the gradients to zero

            # Print the loss every 100 steps
            if steps % 100 == 0 and loss is not None:
                print(f'Loss at step {steps}: {loss.item()}')
                losses.append(loss.item())        


batch_size = 32
print('Vocabulary size:', vocab_size)  # Length of the vocabulary list (this includes the space character)
print('Block size:', block_size) #Max length of the context
print('Batch size:', batch_size)
m = BiagramLM(vocab_size, block_size, batch_size) # Instantiate the model
logits, loss = m.forward(xb, yb) # m(xb, yb) # logits  = [number of predictions (B*T), number of possible tokens (C)], targets = [B*T] = [correct token index for each prediction]
print("Logits Shape:", logits.shape) # (batch_size, block_size, vocab_size) -> 4 times 8 characters, each embedded as a 65-dim vector
if loss is not None:
    print("Current Loss:", loss.item())        # Loss value

# Cross entropy only cares about the probability given to the correct class
#   Loss = −log(pcorrect​)
print("If the model knew absolutely nothing and predicted all 65 characters equally (uniform) pi = 1/65 the L = -ln(1/65):", -np.log(1/65))

m.train()

#Producing The First Text
ix = torch.zeros((1, 1), dtype=torch.long)  # Start with a single tensor of shape (1, 1) holding a 0 (new line)
tokens = m.generate(ix, max_new_tokens=100) # Generate 100 tokens as a sequence of indices
print(tokens.shape)                         # Print the shape of the resulting sequence of tokens
print(decode(tokens[0].tolist()))           # Decode the resulting sequence of indices to a string




Vocabulary size: 65
Block size: 8
Batch size: 32
Embedding Shape: 65 X 65
Logits Shape: torch.Size([32, 65])
Current Loss: 4.878634929656982
If the model knew absolutely nothing and predicted all 65 characters equally (uniform) pi = 1/65 the L = -ln(1/65): 4.174387269895637
Loss at step 0: 4.648484230041504
Loss at step 100: 4.642974853515625
Loss at step 200: 4.47345495223999
Loss at step 300: 4.254473686218262
Loss at step 400: 4.2933831214904785
Loss at step 500: 4.162596225738525
Loss at step 600: 4.020341873168945
Loss at step 700: 4.030514240264893
Loss at step 800: 3.8753597736358643
Loss at step 900: 3.8104541301727295
Loss at step 1000: 3.7026751041412354
Loss at step 1100: 3.619042158126831
Loss at step 1200: 3.6299028396606445
Loss at step 1300: 3.523883104324341
Loss at step 1400: 3.3767645359039307
Loss at step 1500: 3.4229278564453125
Loss at step 1600: 3.435028553009033
Loss at step 1700: 3.298100709915161
Loss at step 1800: 3.3145272731781006
Loss at step 1900: 3.114296

In [6]:
#Producing The First Text
ix = torch.zeros((1, 1), dtype=torch.long)  # Start with a single tensor of shape (1, 1) holding a 0 (new line)
tokens = m.generate(ix, max_new_tokens=500) # Generate 100 tokens as a sequence of indices
print(tokens.shape)                         # Print the shape of the resulting sequence of tokens
print(decode(tokens[0].tolist()))           # Decode the resulting sequence of indices to a string

Input idx: 1 tensor([[0]])
Logits Last Token: tensor([[ 2.3958, -5.1067, -5.3824, -5.9111, -4.4439, -0.8038, -4.1307, -3.8054,
         -4.6960, -2.2339, -6.3244, -5.5257, -4.8153,  1.7830,  0.9912,  0.6306,
          0.2444, -0.3996,  0.6730,  0.3093,  0.7724,  1.3556, -0.8891, -0.1239,
          0.4746,  0.7495,  0.4569,  0.5202,  0.2094, -0.9055, -0.0928,  1.0243,
          1.8925, -0.9193, -1.0754,  1.4438, -4.8891,  0.1990, -6.1381, -1.1367,
         -1.5754, -1.6426, -1.7113, -2.3958, -1.7854, -2.0269, -1.2675, -1.7804,
         -2.9030, -2.8786, -2.0512, -1.5241, -1.8923, -1.5245, -1.3695, -4.2371,
         -2.1926, -1.2332, -0.7037, -2.5695, -2.7896, -1.1604, -3.5063, -1.7953,
         -5.8343]], grad_fn=<SliceBackward0>) Shape: torch.Size([1, 65]) idx_next: tensor([[21]]) logits Shape: torch.Size([1, 1, 65])
Input idx: 8 tensor([[ 0, 21, 63, 53, 58, 43, 52, 45]])
Logits Last Token: tensor([[-7.6062e-01,  2.0126e+00, -1.6945e+00, -5.4842e+00, -5.8462e+00,
         -1.0906e+00, 